# Section 7: Function Calling, Tool Use & Structured Output — Hands-On

### GenAI Masterclass — Turning LLMs into Systems That *Do* Things

---

In **Sections 1–6**, you taught the model to *think*. You learned how LLMs work, how to prompt them, how to ground them in your own data, and how to chain them together with frameworks.

But everything so far has produced one kind of output: **text**.

Real GenAI products do more — they query databases, send emails, generate charts, run analytics, update records. To do that, the LLM needs a way to **take actions**. That's what function calling (also called *tool use*) is for.

In this notebook you'll build it from the ground up:

- The **tool-use loop** — who actually runs the code (spoiler: not the LLM)
- **Your first tool call** end-to-end in OpenAI, then Anthropic
- **Multi-turn** chains and **parallel** tool calls
- How **schema quality** changes model reliability — measured live
- **Structured output**: JSON Mode → Pydantic → `instructor`
- **Reliability**: retries, fallbacks, validation, garbage-output handling
- **Logging** every tool call for production debugging
- **Capstone**: a *Data Analyst Assistant* with 4 tools (SQL, charts, summary, PDF export)

By the end, you'll have built systems where one English sentence triggers SQL execution, chart generation, and PDF export — orchestrated by an LLM through tools you wrote.

---

## Setup

**Python 3.10+ recommended.** We'll use:

- **OpenAI** (`gpt-4o-mini`) — main model, matches Sections 1–6
- **Anthropic** (`claude-haiku-4-5`) — for the provider-comparison chapter
- **instructor + Pydantic** — production-grade structured output
- **tenacity** — retry logic
- **matplotlib + reportlab + sqlite3** — for the capstone (SQL → charts → PDF)

Make sure your `.env` has `OPENAI_API_KEY` and `ANTHROPIC_API_KEY`.

In [ ]:
# Install required libraries (run once)
import sys
import subprocess

pkgs = [
    "openai", "anthropic", "instructor", "pydantic",
    "tenacity", "matplotlib", "reportlab", "python-dotenv",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
print(f"✅  Installed for Python {sys.version_info.major}.{sys.version_info.minor}")

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# Verify keys are loaded
for key in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY"]:
    status = "✅" if os.getenv(key) else "❌ MISSING"
    print(f"{status}  {key}")

In [ ]:
# ── Clients we'll use throughout ──
from openai import OpenAI
from anthropic import Anthropic

oai = OpenAI()
anthropic = Anthropic()

OPENAI_MODEL = "gpt-4o-mini"
ANTHROPIC_MODEL = "claude-haiku-4-5"

print(f"✅  OpenAI client     → {OPENAI_MODEL}")
print(f"✅  Anthropic client  → {ANTHROPIC_MODEL}")

---

## 3. The Tool-Use Loop

Before writing any tool, internalize this: **the LLM never runs your code**. It only returns a structured message saying which function it would like to call and with what arguments. *You* run the function and feed the result back. Every tool call follows this 5-step cycle.

### 3.1 — Visualize the Loop

Here's the cycle in plain Python — no LLM yet, just to see who does what at each stage.

In [ ]:
# A made-up tool the "model" wants to call
def get_weather(city: str) -> dict:
    fake = {
        "Tokyo":  {"temp": 14, "condition": "rain"},
        "Dubai":  {"temp": 38, "condition": "sunny"},
        "London": {"temp": 9,  "condition": "cloudy"},
    }
    return fake.get(city, {"error": "unknown city"})

# Step 1: user asks something
user_question = "Should I bring an umbrella to Tokyo today?"

# Step 2: model "decides" to call a tool — it returns JSON, not an answer
pretend_model_output = {
    "tool_calls": [{"name": "get_weather", "arguments": {"city": "Tokyo"}}]
}

# Step 3: YOUR code runs the actual function
call = pretend_model_output["tool_calls"][0]
result = get_weather(**call["arguments"])
print("Tool result:", result)

# Step 4 + 5: send result back to model, model writes the final answer
print("→ The model would now use this result to write the final reply.")

Notice what just happened: the model said *what* to do, our Python actually *did* it. That separation is the whole safety story behind function calling — the LLM can ask to do anything; only your code decides whether to comply.

💡 **Key takeaway:** The model proposes; your code disposes. Every tool call is just JSON describing a function name and arguments — execution always happens in your runtime.

---

## 4. Your First Real Tool Call (OpenAI)

Now we hook a real LLM into the loop. Same `get_weather` function — but this time the model genuinely decides whether to call it.

### 4.1 — Describe the Tool to the Model

The model can't see your Python code. It only sees a **JSON schema** that describes the function — name, description, and parameter types. This is the menu the LLM reads.

In [ ]:
tools = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": (
            "Get current weather for a specific city. "
            "Use this whenever the user asks about current conditions, "
            "temperature, or whether to bring rain gear."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "The city name, e.g. 'Tokyo' or 'London'."
                }
            },
            "required": ["city"]
        }
    }
}]
print("✅ Tool schema ready")

### 4.2 — Send the Prompt With Tools Attached

In [ ]:
response = oai.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[{"role": "user", "content": "Should I bring an umbrella to Tokyo today?"}],
    tools=tools,
)

msg = response.choices[0].message
print("content:    ", msg.content)        # → None  (the model is NOT answering)
print("tool_calls: ", msg.tool_calls)     # → the function it wants to call

Two things to notice. `content` is `None` — the model isn't answering yet. And `tool_calls` contains the function it wants to invoke, with arguments as a **JSON string** (not a Python dict — you'll need `json.loads`).

### 4.3 — Run the Function and Send the Result Back

In [ ]:
import json

# Pull the tool call out of the response
tool_call = msg.tool_calls[0]
args = json.loads(tool_call.function.arguments)
print("Parsed args:", args)

# Actually run YOUR function
result = get_weather(**args)
print("Function result:", result)

In [ ]:
# Build the conversation: original question + model's tool-call + tool result
messages = [
    {"role": "user", "content": "Should I bring an umbrella to Tokyo today?"},
    msg,  # the model's tool-call message, passed back unchanged
    {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": json.dumps(result),
    },
]

# Call the model again — now it has the data it was missing
final = oai.chat.completions.create(
    model=OPENAI_MODEL,
    messages=messages,
    tools=tools,
)

print(final.choices[0].message.content)

💡 **Key takeaway:** That six-step round-trip — define schema → call model → parse tool call → run function → send result back → call model again — is the entire mechanic. Every fancier tool-use pattern in this notebook is just a variation on this loop.

---

## 5. OpenAI vs Anthropic — Same Idea, Different Syntax

Every major provider supports function calling. The concept is identical; the wire format differs in small ways. Let's run the **same tool** through both APIs side by side.

### 5.1 — Anthropic's Tool Format

In [ ]:
# Anthropic uses a flatter shape: name, description, input_schema (not "parameters")
anthropic_tools = [{
    "name": "get_weather",
    "description": (
        "Get current weather for a specific city. "
        "Use this whenever the user asks about current conditions."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "The city name"}
        },
        "required": ["city"]
    }
}]

response = anthropic.messages.create(
    model=ANTHROPIC_MODEL,
    max_tokens=1024,
    messages=[{"role": "user", "content": "What's the weather in Tokyo?"}],
    tools=anthropic_tools,
)

# Anthropic returns content blocks — iterate and look for type=="tool_use"
for block in response.content:
    if block.type == "tool_use":
        print(f"Tool requested: {block.name}")
        print(f"Arguments:      {block.input}   # ← already a dict, no json.loads needed")

### 5.2 — The Differences, Side by Side

| | OpenAI | Anthropic |
|--|--|--|
| Tool wrapper | `{"type":"function","function":{...}}` | flat — name + description + input_schema |
| Schema field | `"parameters"` | `"input_schema"` |
| Read result | `response.choices[0].message.tool_calls[]` | iterate `response.content`, check `block.type` |
| Arguments | JSON string (needs `json.loads`) | Python dict already |

The mental model is identical. Learn one, you can read the other in five minutes.

💡 **Key takeaway:** When you build production software, write a thin abstraction layer that takes your tool definitions in one format and adapts to whichever provider you're calling. It pays off the day you switch models.

---

## 6. Multi-Turn and Parallel Tool Calls

Real questions rarely fit one tool call. Sometimes the model needs a **chain** (look up X, then use X to find Y). Sometimes it needs **independent lookups** that can run in parallel. Both patterns are just variations on the loop.

### 6.1 — A General-Purpose Multi-Turn Loop

This is the function you'll actually use in production. It wraps the loop in a `while`, capped by `max_turns` so a confused model can't loop forever.

In [ ]:
def chat_with_tools(user_message, tools, tool_functions, max_turns=10, model=OPENAI_MODEL):
    """Run a multi-turn tool-use conversation until the model produces a final answer."""
    messages = [{"role": "user", "content": user_message}]
    
    for turn in range(max_turns):
        response = oai.chat.completions.create(model=model, messages=messages, tools=tools)
        msg = response.choices[0].message
        messages.append(msg)
        
        # No tool calls → the model is done
        if not msg.tool_calls:
            return msg.content, messages
        
        # Otherwise, run each requested tool and add its result
        for tc in msg.tool_calls:
            fn = tool_functions[tc.function.name]
            args = json.loads(tc.function.arguments)
            print(f"  [turn {turn}] → {tc.function.name}({args})")
            result = fn(**args)
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(result),
            })
    
    raise Exception(f"Exceeded {max_turns} turns without a final answer")

# Reuse the get_weather tool from earlier
tool_functions = {"get_weather": get_weather}

answer, history = chat_with_tools(
    "Compare the weather in Tokyo, London, and Dubai. Which is best for outdoor running today?",
    tools=tools,
    tool_functions=tool_functions,
)
print("\n📝 Final answer:\n", answer)

### 6.2 — Parallel Tool Calls

Watch the printout above closely. Modern models (GPT-4o, Claude Sonnet 4) usually return **multiple tool calls in a single response** when the lookups don't depend on each other. That's *parallel tool calling* — and you can actually run those functions concurrently.

In [ ]:
import concurrent.futures

def chat_with_parallel_tools(user_message, tools, tool_functions, max_turns=10):
    """Same loop, but executes independent tool calls concurrently."""
    messages = [{"role": "user", "content": user_message}]
    
    for turn in range(max_turns):
        response = oai.chat.completions.create(model=OPENAI_MODEL, messages=messages, tools=tools)
        msg = response.choices[0].message
        messages.append(msg)
        
        if not msg.tool_calls:
            return msg.content
        
        # Fan out — all tool calls in this turn run together
        def run_one(tc):
            fn = tool_functions[tc.function.name]
            args = json.loads(tc.function.arguments)
            return tc.id, fn(**args)
        
        with concurrent.futures.ThreadPoolExecutor(max_workers=10) as exe:
            futures = [exe.submit(run_one, tc) for tc in msg.tool_calls]
            for future in concurrent.futures.as_completed(futures):
                tc_id, result = future.result()
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc_id,
                    "content": json.dumps(result),
                })
    raise Exception("Exceeded max turns")

answer = chat_with_parallel_tools(
    "Compare the weather in Tokyo, London, and Dubai.",
    tools=tools,
    tool_functions=tool_functions,
)
print(answer)

💡 **Key takeaway:** A 5-tool sequential chain at 800 ms per call = 4 seconds. Parallelized = ~800 ms total. Users notice. Use parallel for independent network-bound calls; *don't* parallelize tool calls that depend on each other's output.

---

## 7. Schema Design — Measured Live

Claim: how you write the schema is the single biggest lever on tool-use reliability. Let's prove it. Same task — *search products* — defined three different ways. We'll send the same 6 user questions through each and count how often the model uses the tool correctly.

### 7.1 — Three Schemas, Same Underlying Function

In [ ]:
# The real function never changes
def search_products(query: str, limit: int = 5, category: str = "none") -> dict:
    catalog = {
        "electronics": ["ProMax 15 laptop", "SoundPods Pro", "TabletAir 11"],
        "accessories": ["leather case", "USB-C cable", "screen protector"],
    }
    if category != "none" and category in catalog:
        items = catalog[category]
    else:
        items = [i for sub in catalog.values() for i in sub]
    matches = [i for i in items if query.lower() in i.lower()][:limit]
    return {"results": matches, "count": len(matches)}

# Schema A — vague (what beginners write)
schema_vague = [{
    "type": "function",
    "function": {
        "name": "do_stuff",
        "description": "does things with data",
        "parameters": {
            "type": "object",
            "properties": {"input": {"type": "string"}}
        }
    }
}]

# Schema B — okay (most tutorials)
schema_okay = [{
    "type": "function",
    "function": {
        "name": "search_products",
        "description": "Search the product catalog",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string"},
                "limit": {"type": "integer"}
            },
            "required": ["query"]
        }
    }
}]

# Schema C — production grade
schema_good = [{
    "type": "function",
    "function": {
        "name": "search_products",
        "description": (
            "Search the TechStore product catalog by keyword. "
            "Use this when the user asks to find, browse, or compare products. "
            "Do NOT use for questions about orders, returns, or shipping."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search keywords, e.g. 'wireless headphones'. Avoid full sentences."
                },
                "limit": {
                    "type": "integer",
                    "description": "Max results. Default 5, max 20.",
                    "default": 5
                },
                "category": {
                    "type": "string",
                    "enum": ["electronics", "accessories", "none"],
                    "description": "Optional category filter. Use 'none' if unspecified."
                }
            },
            "required": ["query"]
        }
    }
}]

print("✅ Three schemas defined")

### 7.2 — Score Each Schema on the Same Test Questions

In [ ]:
test_questions = [
    "Show me wireless headphones",
    "Find laptops under $1500",
    "What accessories do you sell?",
    "Where's my order #4521?",       # ← should NOT call the tool
    "I want to return something",     # ← should NOT call the tool
    "Got any screen protectors?",
]

def score_schema(schema, label):
    correct, wrong, no_call = 0, 0, 0
    for q in test_questions:
        resp = oai.chat.completions.create(
            model=OPENAI_MODEL, messages=[{"role": "user", "content": q}], tools=schema,
        )
        msg = resp.choices[0].message
        # Heuristic: questions 0,1,2,5 should call the tool; 3,4 should NOT
        should_call = q not in test_questions[3:5]
        called = bool(msg.tool_calls)
        if should_call and called:
            correct += 1
        elif not should_call and not called:
            correct += 1
        elif should_call and not called:
            no_call += 1
        else:
            wrong += 1
    print(f"{label:20s}  correct: {correct}/6   wrong-tool: {wrong}   missed: {no_call}")

print("Running 6 questions × 3 schemas — this calls the API 18 times:\n")
score_schema(schema_vague, "vague schema")
score_schema(schema_okay,  "okay schema")
score_schema(schema_good,  "production schema")

Run that a few times — the production schema almost always scores 6/6 while the vague one struggles. The model is the same; the only thing that changed is how clearly we described the menu.

💡 **Key takeaway:** The two highest-leverage things you can do for tool-use reliability are (1) tell the model *when NOT to use* this tool, and (2) describe every parameter with a concrete example. Together they often move accuracy from ~60% to ~95%.

---

## 8. Structured Output — JSON Mode → Pydantic → instructor

Sometimes you don't need the model to *act* — you just need its answer in a specific shape (extract entities, classify text, fill a form). Three tools, in order of how production-ready each is.

### 8.1 — JSON Mode

A simple flag that forces valid JSON. Cheap, available, but doesn't enforce *your* schema — only that the output parses.

In [ ]:
response = oai.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[
        {"role": "system", "content": "Extract entities and return JSON."},
        {"role": "user", "content": "John Smith ordered 3 laptops from Best Buy on Tuesday."},
    ],
    response_format={"type": "json_object"},  # ← the magic flag
)

raw = response.choices[0].message.content
print("Raw response (guaranteed valid JSON):")
print(raw)

import json
data = json.loads(raw)
print("\nParsed:", data)

### 8.2 — Pydantic Models

Valid JSON isn't enough. We want valid JSON *that matches our shape* — right types, required fields, sensible values. Pydantic models give you all of that with type hints.

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Literal

class ActionItem(BaseModel):
    owner: str = Field(description="Person responsible for this action")
    task: str = Field(description="What needs to be done")
    due_date: str = Field(description="ISO date, e.g. '2025-11-30'")
    priority: Literal["high", "medium", "low"]

class MeetingSummary(BaseModel):
    title: str
    attendees: List[str]
    key_decisions: List[str]
    action_items: List[ActionItem]

# The schema is automatically derived from the type hints
print(MeetingSummary.model_json_schema())

### 8.3 — `instructor`: the Production Standard

`instructor` wraps the OpenAI client and lets you pass a Pydantic model as `response_model`. It handles the schema, calls the LLM, parses the response into a typed instance, and **auto-retries on validation failures with the error fed back to the model**.

In [ ]:
import instructor

# Wrap the OpenAI client
client = instructor.from_openai(OpenAI())

transcript = """
Sarah: The launch is still on track for Nov 15. Mike, can you finish the landing page by Nov 8?
Mike: Sure, I'll have it ready by then. High priority.
Sarah: Priya, we need legal sign-off on the terms by Nov 7 — that's blocking us.
Priya: I'll talk to legal tomorrow, should have an answer by end of week.
Sarah: Decision: we're using Stripe for payments, not PayPal.
"""

summary = client.chat.completions.create(
    model=OPENAI_MODEL,
    response_model=MeetingSummary,   # ← typed output
    max_retries=2,                    # ← auto-retry on validation failures
    messages=[
        {"role": "system", "content": "Extract structured meeting notes from this transcript."},
        {"role": "user", "content": transcript},
    ],
)

# summary is a real MeetingSummary instance — not a dict, not a string
print(f"Title:     {summary.title}")
print(f"Attendees: {summary.attendees}")
print(f"\nDecisions:")
for d in summary.key_decisions:
    print(f"  • {d}")
print(f"\nAction items:")
for a in summary.action_items:
    print(f"  • [{a.priority.upper()}] {a.owner} — {a.task} (due {a.due_date})")

Notice what we got for free: full IDE autocomplete on `summary.action_items[0].owner`, type checking, and automatic retry if the LLM had returned bad data. In real codebases you'll see `instructor.from_openai(client)` as the very first line after imports.

💡 **Key takeaway:** JSON Mode gives you valid JSON. Pydantic gives you a validated shape. `instructor` gives you both, plus auto-retry, plus IDE-friendly typed objects — for one extra import.

---

## 9. Validation — When the JSON Is *Almost* Right

Pydantic catches type errors and missing fields. But sometimes the output is structurally valid yet **semantically wrong** — a negative age, an email without an `@`, a due date before the issue date. For those, use field validators and post-validation business rules.

In [ ]:
from pydantic import BaseModel, field_validator, EmailStr, ValidationError

class Customer(BaseModel):
    name: str
    age: int
    email: EmailStr   # built-in validation
    
    @field_validator("age")
    @classmethod
    def age_must_be_realistic(cls, v):
        if v < 0 or v > 130:
            raise ValueError(f"Age {v} is not realistic")
        return v
    
    @field_validator("name")
    @classmethod
    def name_must_have_content(cls, v):
        if len(v.strip()) < 2:
            raise ValueError("Name too short")
        return v.strip()

# Good input
try:
    c = Customer(name="Nij", age=33, email="nij@example.com")
    print(f"✅ Valid: {c}")
except ValidationError as e:
    print(f"❌ {e}")

# Bad input — caught by our validator
try:
    c = Customer(name="X", age=250, email="not-an-email")
    print(f"✅ Valid: {c}")
except ValidationError as e:
    print(f"❌ Validation failed:\n{e}")

Now combine it with `instructor` — the LLM will be told exactly what went wrong and asked to retry.

In [ ]:
# Try to extract a customer from a bad input — watch instructor retry
try:
    bad_input = "Customer X is 250 years old, email not-an-email"
    c = client.chat.completions.create(
        model=OPENAI_MODEL,
        response_model=Customer,
        max_retries=2,
        messages=[{"role": "user", "content": bad_input}],
    )
    print(f"Recovered: {c}")
except Exception as e:
    print(f"Couldn't recover after retries: {e}")

💡 **Key takeaway:** Don't trust the LLM with anything you wouldn't trust an unverified user with. Validate types *and* business rules. Send errors back to the model — it almost always fixes itself on the next try.

---

## 10. Reliability — Retries, Fallbacks, Garbage Handling

Production systems plan for failure. Networks blip, rate limits hit, models occasionally return weird output. Three layered defenses: **retry** for transient issues, **fallback** for systematic ones, **error-as-message** for everything else.

### 10.1 — Exponential-Backoff Retries with `tenacity`

In [ ]:
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from openai import APIError, RateLimitError

@retry(
    retry=retry_if_exception_type((APIError, RateLimitError)),
    wait=wait_exponential(multiplier=1, min=2, max=30),
    stop=stop_after_attempt(5),
    reraise=True,
)
def call_llm_with_retry(prompt: str) -> str:
    response = oai.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content

# Backoff schedule: 2s, 4s, 8s, 16s, 30s — gives rate limits time to reset
print(call_llm_with_retry("Say hi in three words."))

### 10.2 — Fallback Chain

If the best model + structured output fails, try a smaller model. If that fails, return a graceful error. Each tier is *less ambitious* than the last.

In [ ]:
def robust_summary(document: str) -> str:
    # Tier 1: best model + structured output
    try:
        result = client.chat.completions.create(
            model=OPENAI_MODEL,
            response_model=MeetingSummary,
            max_retries=1,
            messages=[{"role": "user", "content": document}],
        )
        return f"[tier 1] structured: {len(result.action_items)} action items"
    except Exception as e:
        print(f"  Tier 1 failed: {type(e).__name__}")
    
    # Tier 2: cheaper model, unstructured prose
    try:
        response = oai.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[
                {"role": "system", "content": "Summarize this document in 3 bullets."},
                {"role": "user", "content": document},
            ],
        )
        return f"[tier 2] prose: {response.choices[0].message.content[:80]}..."
    except Exception:
        print("  Tier 2 failed")
    
    # Tier 3: graceful failure
    return "[tier 3] Summary unavailable. Please review the document directly."

result = robust_summary("Sarah: launch is on track for Nov 15. Mike will finish landing page by Nov 8. Priya needs legal sign-off.")
print("Result:", result)

### 10.3 — Convert Errors to Messages, Not Exceptions

When a tool fails or returns garbage, **never crash the loop**. Convert the error into a message the model can read. Modern models are excellent at fixing their own mistakes when given specific feedback.

In [ ]:
def safely_execute_tool(tc, tool_functions, registered_tool_names):
    """Run a tool call, return either its result or a model-readable error."""
    # 1. Tool name exists?
    if tc.function.name not in registered_tool_names:
        return {"error": f"Unknown tool '{tc.function.name}'. Available: {registered_tool_names}"}
    
    # 2. Arguments parse?
    try:
        args = json.loads(tc.function.arguments)
    except json.JSONDecodeError as e:
        return {"error": f"Malformed arguments JSON: {e}"}
    
    # 3. Function runs?
    try:
        return tool_functions[tc.function.name](**args)
    except TypeError as e:
        return {"error": f"Wrong arguments: {e}"}
    except Exception as e:
        return {"error": f"Tool execution failed: {e}"}

# Demonstrate each failure mode
class FakeCall:
    def __init__(self, name, args):
        self.function = type("f", (), {"name": name, "arguments": args})()

print(safely_execute_tool(FakeCall("hallucinated_tool", '{}'), {"get_weather": get_weather}, ["get_weather"]))
print(safely_execute_tool(FakeCall("get_weather", '{not valid json}'), {"get_weather": get_weather}, ["get_weather"]))
print(safely_execute_tool(FakeCall("get_weather", '{"city": "Tokyo"}'), {"get_weather": get_weather}, ["get_weather"]))

💡 **Key takeaway:** Crashing kills the loop; messages let the model self-correct. In production, every possible failure inside a tool call becomes a structured error message that goes back into the conversation.

---

## 11. Logging — You Can't Debug What You Can't See

Tool-using systems are multi-step by nature. When something goes wrong in production, you need to trace which tools were called, with what arguments, in what order, and how long each took.

In [ ]:
import time
import uuid
import logging

logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.getLogger("tool_use")

def logged_chat_with_tools(user_message, tools, tool_functions, max_turns=10):
    request_id = str(uuid.uuid4())[:8]
    messages = [{"role": "user", "content": user_message}]
    log.info(f"[{request_id}] ── new request: {user_message[:60]!r}")
    
    for turn in range(max_turns):
        response = oai.chat.completions.create(model=OPENAI_MODEL, messages=messages, tools=tools)
        msg = response.choices[0].message
        messages.append(msg)
        
        usage = response.usage
        log.info(f"[{request_id}] turn={turn} tokens_in={usage.prompt_tokens} tokens_out={usage.completion_tokens}")
        
        if not msg.tool_calls:
            log.info(f"[{request_id}] ── done")
            return msg.content
        
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            start = time.time()
            try:
                result = tool_functions[tc.function.name](**args)
                status = "ok"
            except Exception as e:
                result = {"error": str(e)}
                status = "error"
            latency_ms = int((time.time() - start) * 1000)
            log.info(f"[{request_id}] turn={turn} tool={tc.function.name} status={status} latency_ms={latency_ms}")
            
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(result),
            })
    raise Exception("max turns")

answer = logged_chat_with_tools(
    "What's the weather in Tokyo and London?",
    tools=tools,
    tool_functions={"get_weather": get_weather},
)
print("\n📝", answer)

💡 **Key takeaway:** From day one, log `request_id`, `turn`, `tool`, `args`, `status`, `latency_ms`, and token usage. When something breaks in production at 2 AM, these logs are the difference between a 5-minute fix and a 5-hour mystery.

---

## 12. Capstone — Data Analyst Assistant

Everything in this notebook comes together now. You'll build a system where a single English sentence triggers SQL execution, chart generation, analytical writing, and PDF export — orchestrated by an LLM through four tools you wrote.

The architecture:

| Tool | Purpose |
|------|---------|
| `run_sql` | Execute a SELECT against the sales database |
| `make_chart` | Render a chart from data and save to disk |
| `summarize_data` | Turn numbers into an analytical narrative |
| `export_report` | Bundle everything into a PDF |

The model decides the order, runs them in parallel when it can, and combines the results into a finished analysis.

### 12.1 — Set Up a Tiny Sales Database

In [ ]:
import sqlite3
import os

DB_PATH = "/tmp/analytics.db"
CHART_DIR = "/tmp/charts"
REPORT_DIR = "/tmp/reports"
os.makedirs(CHART_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

# Fresh DB each run
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)
conn.executescript("""
CREATE TABLE sales (
    product_name TEXT,
    quarter TEXT,
    revenue INTEGER,
    units INTEGER
);
INSERT INTO sales VALUES
    ('ProMax 15',     'Q2-2025',  980000, 1200),
    ('ProMax 15',     'Q3-2025', 1200000, 1450),
    ('SoundPods Pro', 'Q2-2025',  920000, 4500),
    ('SoundPods Pro', 'Q3-2025',  890000, 4300),
    ('TabletAir 11',  'Q2-2025',  580000,  900),
    ('TabletAir 11',  'Q3-2025',  640000, 1000),
    ('SmartHub X',    'Q2-2025',  470000, 2100),
    ('SmartHub X',    'Q3-2025',  520000, 2400),
    ('WatchFit 3',    'Q2-2025',  390000, 3000),
    ('WatchFit 3',    'Q3-2025',  410000, 3200);
""")
conn.commit()
conn.close()
print(f"✅ Sales DB ready at {DB_PATH}")

### 12.2 — The Four Tool Functions

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

def run_sql(query: str, max_rows: int = 100) -> dict:
    """Execute a read-only SELECT against the sales database."""
    if not query.strip().lower().startswith("select"):
        return {"error": "Only SELECT queries are allowed."}
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    try:
        cur = conn.execute(query)
        rows = [dict(r) for r in cur.fetchmany(max_rows)]
        return {"rows": rows, "count": len(rows)}
    except sqlite3.Error as e:
        return {"error": f"SQL error: {e}"}
    finally:
        conn.close()

def make_chart(type: str, title: str, labels: list, series: list) -> dict:
    """Render a chart and save to disk."""
    fig, ax = plt.subplots(figsize=(10, 5))
    if type == "bar_grouped":
        x = np.arange(len(labels))
        width = 0.8 / max(1, len(series))
        for i, s in enumerate(series):
            ax.bar(x + i * width, s["data"], width, label=s["name"])
        ax.set_xticks(x + width * (len(series) - 1) / 2)
        ax.set_xticklabels(labels, rotation=30, ha="right")
    elif type == "line":
        for s in series:
            ax.plot(labels, s["data"], marker="o", label=s["name"])
    elif type == "bar":
        ax.bar(labels, series[0]["data"])
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    path = f"{CHART_DIR}/{uuid.uuid4().hex[:8]}.png"
    plt.savefig(path, dpi=120)
    plt.close()
    return {"chart_path": path}

def summarize_data(context: str, data: list) -> dict:
    """Use a sub-LLM call to write an analytical narrative."""
    prompt = (
        f"Context: {context}\n\nData: {json.dumps(data)}\n\n"
        "Write a 4-5 bullet analytical summary. Call out trends, outliers, "
        "and percentage changes. Be concise — this is for executives."
    )
    resp = oai.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    return {"summary": resp.choices[0].message.content}

def export_report(title: str, summary: str, chart_paths: list) -> dict:
    """Bundle summary + charts into a PDF."""
    path = f"{REPORT_DIR}/{uuid.uuid4().hex[:8]}.pdf"
    c = canvas.Canvas(path, pagesize=letter)
    c.setFont("Helvetica-Bold", 16)
    c.drawString(72, 720, title)
    c.setFont("Helvetica", 10)
    y = 690
    for line in summary.split("\n"):
        c.drawString(72, y, line[:90])
        y -= 14
        if y < 100:
            c.showPage()
            y = 720
    for cp in chart_paths:
        c.showPage()
        c.drawImage(cp, 72, 300, width=470, height=240)
    c.save()
    return {"report_path": path}

print("✅ Tool functions ready")

### 12.3 — Tool Schemas

In [ ]:
ANALYST_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "run_sql",
            "description": (
                "Execute a SELECT query against the sales database. "
                "Use whenever the user needs data, comparisons, or aggregates. "
                "Schema: sales(product_name TEXT, quarter TEXT, revenue INTEGER, units INTEGER). "
                "Quarters look like 'Q3-2025'."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "A valid SELECT statement."},
                    "max_rows": {"type": "integer", "description": "Max rows to return. Default 100.", "default": 100},
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "make_chart",
            "description": "Create a chart from query results. Use after run_sql when a visualization helps.",
            "parameters": {
                "type": "object",
                "properties": {
                    "type": {"type": "string", "enum": ["bar", "bar_grouped", "line"]},
                    "title": {"type": "string"},
                    "labels": {"type": "array", "items": {"type": "string"}},
                    "series": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "name": {"type": "string"},
                                "data": {"type": "array", "items": {"type": "number"}},
                            },
                        },
                    },
                },
                "required": ["type", "title", "labels", "series"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "summarize_data",
            "description": "Generate an analytical narrative from raw numbers. Use to turn query results into executive-friendly bullets.",
            "parameters": {
                "type": "object",
                "properties": {
                    "context": {"type": "string", "description": "What this data represents."},
                    "data": {"type": "array", "description": "The rows or numbers to summarize."},
                },
                "required": ["context", "data"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "export_report",
            "description": "Bundle summary + charts into a PDF. Use as the final step to package an analysis.",
            "parameters": {
                "type": "object",
                "properties": {
                    "title": {"type": "string"},
                    "summary": {"type": "string"},
                    "chart_paths": {"type": "array", "items": {"type": "string"}},
                },
                "required": ["title", "summary", "chart_paths"],
            },
        },
    },
]

ANALYST_TOOL_FUNCS = {
    "run_sql": run_sql,
    "make_chart": make_chart,
    "summarize_data": summarize_data,
    "export_report": export_report,
}

print(f"✅ {len(ANALYST_TOOLS)} tools defined")

### 12.4 — The Orchestrator

The same multi-turn loop from chapter 6, with logging from chapter 11 — only the system prompt is new. We give the LLM clear instructions about the workflow.

In [ ]:
ANALYST_SYSTEM_PROMPT = """You are a senior data analyst at TechStore.
You help executives understand sales data by running SQL, creating charts, and writing concise summaries.

Your workflow:
1. Use run_sql to fetch the data you need.
2. If multiple queries are independent, request them in one turn.
3. Use make_chart when visualization helps.
4. Use summarize_data to write the analytical narrative.
5. Use export_report at the very end to package everything.

Always write the final answer in clear, executive-friendly prose."""

def ask_analyst(question: str, max_turns: int = 12) -> str:
    request_id = str(uuid.uuid4())[:8]
    messages = [
        {"role": "system", "content": ANALYST_SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    log.info(f"[{request_id}] ── analyst question: {question[:80]!r}")
    
    for turn in range(max_turns):
        response = oai.chat.completions.create(
            model=OPENAI_MODEL, messages=messages, tools=ANALYST_TOOLS,
        )
        msg = response.choices[0].message
        messages.append(msg)
        
        if not msg.tool_calls:
            log.info(f"[{request_id}] ── final answer produced")
            return msg.content
        
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            log.info(f"[{request_id}] turn={turn} → {tc.function.name}()")
            try:
                result = ANALYST_TOOL_FUNCS[tc.function.name](**args)
            except Exception as e:
                result = {"error": str(e)}
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(result),
            })
    
    raise Exception(f"Exceeded {max_turns} turns")

print("✅ Orchestrator ready")

### 12.5 — Run It

In [ ]:
answer = ask_analyst(
    "What were our top 5 products by revenue in Q3-2025, "
    "and how do they compare to Q2-2025? Make a chart and export a PDF report."
)

print("\n" + "=" * 60)
print("📝 FINAL ANSWER")
print("=" * 60)
print(answer)

Open the chart in `/tmp/charts/` and the PDF in `/tmp/reports/`. The LLM picked the SQL queries, decided when to chart, when to summarize, and when to export — all from one English sentence.

💡 **Key takeaway:** This is the core pattern behind every "AI analyst", "AI assistant", and "AI copilot" you see in industry. The LLM is the conductor; your tools are the orchestra. Build narrow, well-described tools and the model will compose them in ways you didn't have to plan for.

---

## 13. Practice Challenges

Three exercises that put everything in this notebook to work. The scaffolding is here; the implementation is yours.

### 13.1 — Meeting Notes Processor

Build a function that takes a raw meeting transcript and returns a `MeetingSummary` (from chapter 8). 

**Stretch:**
- Add a `field_validator` that converts relative dates ("next Friday") to ISO format.
- Use `max_retries=2` and confirm it auto-fixes when the model invents an enum value.
- Test on 3 different transcripts.

### 13.2 — Multi-Tool Assistant (≥ 4 tools)

Pick a domain you care about. Define at least four tools with proper schemas, then wire them into the multi-turn loop. Ideas:

- **E-commerce concierge**: `search_products`, `get_order_status`, `create_return`, `recommend_alternatives`
- **Calendar copilot**: `find_free_slot`, `create_event`, `list_attendee_conflicts`, `send_invite`
- **Personal finance**: `get_balance`, `list_transactions`, `categorize_spending`, `generate_budget_chart`

**Requirements:**
- Proper descriptions (including "do NOT use for …")
- Pydantic validation on every tool call
- Logging with request_id + turn + tool + latency
- At least one fallback path for a tool that can fail

### 13.3 — Convert Old Projects

Pick 3 things you built earlier in the course that used **string parsing of LLM output**. Rewrite each to use function calling or structured output. Then run both versions on the same 50 inputs and count how many failed in each.

You'll typically see ~50% fewer lines of code and ~3× higher reliability.

---

## Wrap-Up

You started this notebook with an LLM that could only produce text. You're ending it with a system where one English sentence triggers SQL, charts, summaries, and PDF export — coordinated by the LLM through tools you wrote.

Everything in this section is a variation on one loop:

1. **Define tools** as JSON schemas the model can read.
2. **Call the model** with the tools attached.
3. **Run the functions** the model asks for.
4. **Send the results back**.
5. **Repeat** until the model has a final answer.

The skills around that loop — schema design, structured output, validation, retries, fallbacks, logging — are what separate a demo from a production system. You now have all of them in your toolkit.

Next stop: longer-running agent workflows that string many of these patterns together.

---